In [1]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git clone https://{GITHUB_TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
data_dir = "data"
while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

%pip install -q dagshub
!pip install mlflow --quiet

import dagshub
import mlflow

dagshub.init(repo_owner="amama22", repo_name="walmart-forecasting", mlflow=True)
mlflow.set_experiment("XGBoost_Training")

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 31 (delta 11), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 183.79 KiB | 5.93 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/walmart-forecasting
100% 2.70M/2.70M [00:00<00:00, 75.6MB/s]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=75d98f4c-bc19-495e-9986-93c54f07eb27&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c0a3bb25c22e96ce2525e79c570178dd48ccb46e38fced0e2b3004e67b88ed3c




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

2026/07/11 17:20:47 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/450543a787b34d62bae316d13692ca1f', creation_time=1783790447420, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783790447420, lifecycle_stage='active', name='XGBoost_Training', tags={}, trace_location=None, workspace='default'>

In [2]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
import mlflow

from src.data_prep import load_raw_data, merge_all, clean_data
from src.feature_engineering import add_holiday_proximity
from src.evaluation import wmae, walk_forward_splits

train, test, features, stores = load_raw_data(data_dir="data")

train_merged = merge_all(train, features, stores)
test_merged = merge_all(test, features, stores)

print("train_merged shape:", train_merged.shape)
print("test_merged shape:", test_merged.shape)
train_merged.head()

train_merged shape: (421570, 16)
test_merged shape: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):

    null_counts_before = train_merged.isnull().sum()
    null_counts_before = null_counts_before[null_counts_before > 0]
    print("Nulls before cleaning:\n", null_counts_before)

    mlflow.log_param("markdown_fill_strategy", "fillna_0")
    mlflow.log_param("cpi_unemployment_fill_strategy", "ffill_per_store")

    for col, n in null_counts_before.items():
        mlflow.log_metric(f"nulls_before_{col}", int(n))

    train_merged = clean_data(train_merged)
    test_merged = clean_data(test_merged)

    null_counts_after = train_merged.isnull().sum()
    null_counts_after = null_counts_after[null_counts_after > 0]
    print("\nNulls after cleaning:\n", null_counts_after)

    for col in null_counts_before.index:
        mlflow.log_metric(f"nulls_after_{col}", int(train_merged[col].isnull().sum()))

print("train_merged shape:", train_merged.shape)
print("test_merged shape:", test_merged.shape)

Nulls before cleaning:
 MarkDown1    270889
MarkDown2    310322
MarkDown3    284479
MarkDown4    286603
MarkDown5    270138
dtype: int64

Nulls after cleaning:
 Series([], dtype: int64)
🏃 View run XGBoost_Cleaning at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/13f65cd60aca4167bb93c1348b04cba9
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1
train_merged shape: (421570, 16)
test_merged shape: (115064, 15)


In [4]:
train_fe = train_merged.copy()

train_fe["Year"] = train_fe["Date"].dt.year
train_fe["Month"] = train_fe["Date"].dt.month
train_fe["WeekOfYear"] = train_fe["Date"].dt.isocalendar().week.astype(int)

prior_year = train_fe[["Store", "Dept", "Year", "WeekOfYear", "Weekly_Sales"]].copy()
prior_year["Year"] = prior_year["Year"] + 1
prior_year = prior_year.rename(columns={"Weekly_Sales": "sales_lag_52wk"})

train_fe = train_fe.merge(
    prior_year[["Store", "Dept", "Year", "WeekOfYear", "sales_lag_52wk"]],
    on=["Store", "Dept", "Year", "WeekOfYear"],
    how="left",
)

super_bowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])

train_fe["IsSuperBowl"] = train_fe["Date"].isin(super_bowl)
train_fe["IsLaborDay"] = train_fe["Date"].isin(labor_day)
train_fe["IsThanksgiving"] = train_fe["Date"].isin(thanksgiving)
train_fe["IsChristmas"] = train_fe["Date"].isin(christmas)

train_fe = add_holiday_proximity(train_fe)

from src.feature_engineering import add_expanding_dept_avg, add_expanding_store_avg
train_fe = add_expanding_dept_avg(train_fe)
train_fe = add_expanding_store_avg(train_fe)

for col in ["Store", "Dept", "Type"]:
    train_fe[col] = train_fe[col].astype("category")

print("train_fe shape:", train_fe.shape)
train_fe.head()

train_fe shape: (421570, 27)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Month,WeekOfYear,sales_lag_52wk,IsSuperBowl,IsLaborDay,IsThanksgiving,IsChristmas,days_to_nearest_holiday,dept_avg_sales_expanding,store_avg_sales_expanding
0,1,1,2010-02-05,24924.50,False,42.31,2.572,0.0,0.0,0.0,...,2,5,NaN,False,False,False,False,7,NaN,NaN
1,1,1,2010-02-12,46039.49,True,38.51,2.548,0.0,0.0,0.0,...,2,6,NaN,True,False,False,False,0,19596.298000,22516.313699
2,1,1,2010-02-19,41595.55,False,39.93,2.514,0.0,0.0,0.0,...,2,7,NaN,False,False,False,False,-7,25989.064556,22660.639072
3,1,1,2010-02-26,19403.54,False,46.63,2.561,0.0,0.0,0.0,...,2,8,NaN,False,False,False,False,-14,25609.430889,22467.677965
4,1,1,2010-03-05,21827.90,False,46.50,2.625,0.0,0.0,0.0,...,3,9,NaN,False,False,False,False,-21,22992.581944,21745.645939


In [5]:
PRUNED_FEATURE_COLS = [
    "Store", "Dept", "Type", "Size",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown3",
    "Month", "WeekOfYear",
    "sales_lag_52wk",
    "IsThanksgiving",
    "days_to_nearest_holiday",
    "dept_avg_sales_expanding", "store_avg_sales_expanding",
]
TARGET = "Weekly_Sales"

splits = walk_forward_splits(train_fe, n_splits=3, horizon_weeks=38)
print(f"Number of CV folds: {len(splits)}")

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_param("n_splits", len(splits))
    mlflow.log_param("n_features", len(PRUNED_FEATURE_COLS))
    mlflow.log_param("carried_over_from", "LightGBM_Feature_Selection")

    fold_wmaes = []
    importances = np.zeros(len(PRUNED_FEATURE_COLS))

    for i, (train_mask, val_mask) in enumerate(splits):
        X_train, y_train = train_fe.loc[train_mask, PRUNED_FEATURE_COLS], train_fe.loc[train_mask, TARGET]
        X_val, y_val = train_fe.loc[val_mask, PRUNED_FEATURE_COLS], train_fe.loc[val_mask, TARGET]

        model = XGBRegressor(random_state=42, enable_categorical=True, tree_method="hist", verbosity=0)
        model.fit(X_train, y_train)

        preds = model.predict(X_val)
        fold_wmae = wmae(y_val, preds, train_fe.loc[val_mask, "IsHoliday"])
        fold_wmaes.append(fold_wmae)
        importances += model.feature_importances_

        mlflow.log_metric(f"fold{i}_wmae", fold_wmae)
        print(f"Fold {i}: WMAE = {fold_wmae:.2f}")

    mean_wmae = np.mean(fold_wmaes)
    mlflow.log_metric("mean_wmae", mean_wmae)
    print(f"\nMean WMAE (XGBoost, pruned features): {mean_wmae:.2f}")

    importance_df = pd.DataFrame({"feature": PRUNED_FEATURE_COLS, "importance": importances}).sort_values("importance", ascending=False)
    print("\nFeature importances (summed across folds):")
    print(importance_df.to_string(index=False))

    importance_df.to_csv("feature_importance_xgboost.csv", index=False)
    mlflow.log_artifact("feature_importance_xgboost.csv")

Number of CV folds: 3
Fold 0: WMAE = 3753.98
Fold 1: WMAE = 2539.97
Fold 2: WMAE = 1873.04

Mean WMAE (XGBoost, pruned features): 2722.33

Feature importances (summed across folds):
                  feature  importance
                     Dept    1.032488
                    Store    0.543268
 dept_avg_sales_expanding    0.276857
           sales_lag_52wk    0.260352
store_avg_sales_expanding    0.228899
           IsThanksgiving    0.185537
               WeekOfYear    0.133348
             Unemployment    0.062457
  days_to_nearest_holiday    0.055565
                MarkDown3    0.053102
                    Month    0.048689
                      CPI    0.037940
               Fuel_Price    0.030337
                     Size    0.026088
              Temperature    0.025072
                     Type    0.000000
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/78158517a0644adb8c52afe54fd9501f
🧪 View experiment at: 

In [7]:
%pip install -q optuna
import optuna

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "enable_categorical": True,
        "tree_method": "hist",
        "verbosity": 0,
    }

    fold_wmaes = []
    for train_mask, val_mask in splits:
        X_train, y_train = train_fe.loc[train_mask, PRUNED_FEATURE_COLS], train_fe.loc[train_mask, TARGET]
        X_val, y_val = train_fe.loc[val_mask, PRUNED_FEATURE_COLS], train_fe.loc[val_mask, TARGET]

        model = XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        fold_wmaes.append(wmae(y_val, preds, train_fe.loc[val_mask, "IsHoliday"]))

    mean_wmae = np.mean(fold_wmaes)

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_wmae", mean_wmae)

    return mean_wmae

with mlflow.start_run(run_name="XGBoost_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_mean_wmae", study.best_value)
    print("Best WMAE:", study.best_value)
    print("Best params:", study.best_params)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.0 MB/s eta 0:00:00


[I 2026-07-11 17:30:22,148] A new study created in memory with name: no-name-1341c5be-6d36-4fc3-9d99-f1ca357d11b3
[I 2026-07-11 17:33:29,650] Trial 0 finished with value: 2472.6119287720926 and parameters: {'max_depth': 9, 'learning_rate': 0.020023081605965175, 'n_estimators': 581, 'min_child_weight': 15, 'subsample': 0.8724543438972213, 'colsample_bytree': 0.8783493819406509, 'reg_alpha': 0.0008190345023214729, 'reg_lambda': 0.04340798305415044}. Best is trial 0 with value: 2472.6119287720926.


🏃 View run trial_0 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/2ea640e2ac3644bbabd2e7d7fa63c059
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


[I 2026-07-11 17:34:43,218] Trial 1 finished with value: 3029.76751273762 and parameters: {'max_depth': 3, 'learning_rate': 0.08095506829532301, 'n_estimators': 746, 'min_child_weight': 9, 'subsample': 0.6066823500537273, 'colsample_bytree': 0.6638388438729378, 'reg_alpha': 0.0007338226146374081, 'reg_lambda': 1.1660977846322125e-05}. Best is trial 0 with value: 2472.6119287720926.


🏃 View run trial_1 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/2dd3112367f74cfb9f0404ee48f4bd0c
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


...

🏃 View run trial_27 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/641f2789d7dd407c99d3d14d1689577c
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


[I 2026-07-11 18:42:00,140] Trial 28 finished with value: 2533.194169633403 and parameters: {'max_depth': 10, 'learning_rate': 0.10285384913395426, 'n_estimators': 396, 'min_child_weight': 9, 'subsample': 0.8491093699983013, 'colsample_bytree': 0.7564052698321643, 'reg_alpha': 0.004608063074408891, 'reg_lambda': 0.03687904424003708}. Best is trial 12 with value: 2432.060268064685.


🏃 View run trial_28 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/352ea3de289c4f998432d846d114276f
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


[I 2026-07-11 18:43:40,041] Trial 29 finished with value: 2734.1463580071 and parameters: {'max_depth': 11, 'learning_rate': 0.1961444105446814, 'n_estimators': 243, 'min_child_weight': 6, 'subsample': 0.8913699203751495, 'colsample_bytree': 0.7093843550309495, 'reg_alpha': 1.393637676604732, 'reg_lambda': 0.0002034864640808848}. Best is trial 12 with value: 2432.060268064685.


🏃 View run trial_29 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/4a64295b9e1046cd8d26b195ecf848ee
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1
Best WMAE: 2432.060268064685
Best params: {'max_depth': 11, 'learning_rate': 0.028943727780765066, 'n_estimators': 385, 'min_child_weight': 7, 'subsample': 0.8404095548467326, 'colsample_bytree': 0.8351962028504205, 'reg_alpha': 0.0034624838965255007, 'reg_lambda': 0.004336353201066685}
🏃 View run XGBoost_HPO at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/f08314473a5243cd9b93d2f5af32b2a0
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class WalmartFeatureBuilder(BaseEstimator, TransformerMixin):


    def __init__(self, features_df, stores_df):
        self.features_df = features_df
        self.stores_df = stores_df

    def fit(self, X, y=None):
        df = X.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df["Year"] = df["Date"].dt.year
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

        if y is not None:
            df["Weekly_Sales"] = np.asarray(y)

        self.history_ = df[["Store", "Dept", "Year", "WeekOfYear", "Weekly_Sales"]].copy()

        dept_weekly = df.groupby(["Dept", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Dept", "Date"])
        self.dept_avg_final_ = (
            dept_weekly.groupby("Dept")
            .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean(), include_groups=False)
        ).to_dict()

        store_weekly = df.groupby(["Store", "Date"])["Weekly_Sales"].mean().reset_index().sort_values(["Store", "Date"])
        self.store_avg_final_ = (
            store_weekly.groupby("Store")
            .apply(lambda g: g["Weekly_Sales"].iloc[:-1].mean() if len(g) > 1 else g["Weekly_Sales"].mean(), include_groups=False)
        ).to_dict()

        return self

    def transform(self, X):
        df = merge_all(X, self.features_df, self.stores_df)
        df = clean_data(df)

        df["Date"] = pd.to_datetime(df["Date"])
        df["Month"] = df["Date"].dt.month
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
        df["Year"] = df["Date"].dt.year

        thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
        df["IsThanksgiving"] = df["Date"].isin(thanksgiving)

        lookup = self.history_.copy()
        lookup["Year"] = lookup["Year"] + 1
        lookup = lookup.rename(columns={"Weekly_Sales": "sales_lag_52wk"})
        df = df.merge(lookup, on=["Store", "Dept", "Year", "WeekOfYear"], how="left")

        df = add_holiday_proximity(df)

        df["dept_avg_sales_expanding"] = df["Dept"].map(self.dept_avg_final_)
        df["store_avg_sales_expanding"] = df["Store"].map(self.store_avg_final_)

        for col in ["Store", "Dept", "Type"]:
            df[col] = df[col].astype("category")

        return df[PRUNED_FEATURE_COLS]

In [9]:
train_mask, val_mask = splits[-1]

X_train_raw = train.loc[train_mask, ["Store", "Dept", "Date", "IsHoliday"]]
y_train_raw = train.loc[train_mask, "Weekly_Sales"]

X_val_raw = train.loc[val_mask, ["Store", "Dept", "Date", "IsHoliday"]]
y_val_raw = train.loc[val_mask, "Weekly_Sales"]

best_params = study.best_params

check_pipeline = Pipeline([
    ("features", WalmartFeatureBuilder(features, stores)),
    ("model", XGBRegressor(**best_params, random_state=42, enable_categorical=True, tree_method="hist", verbosity=0)),
])

check_pipeline.fit(X_train_raw, y_train_raw)
val_preds = check_pipeline.predict(X_val_raw)

check_wmae = wmae(y_val_raw, val_preds, X_val_raw["IsHoliday"])
print(f"Pipeline WMAE on held-out fold: {check_wmae:.2f}")
print(f"(for reference, flat-DataFrame tuned CV mean was: {study.best_value:.2f})")

Pipeline WMAE on held-out fold: 1615.19
(for reference, flat-DataFrame tuned CV mean was: 2432.06)


In [10]:
X_full = train[["Store", "Dept", "Date", "IsHoliday"]]
y_full = train["Weekly_Sales"]

final_pipeline = Pipeline([
    ("features", WalmartFeatureBuilder(features, stores)),
    ("model", XGBRegressor(**best_params, random_state=42, enable_categorical=True, tree_method="hist", verbosity=0)),
])

with mlflow.start_run(run_name="XGBoost_Final_Pipeline"):
    final_pipeline.fit(X_full, y_full)

    mlflow.log_params(best_params)
    mlflow.log_param("n_features", len(PRUNED_FEATURE_COLS))
    mlflow.log_param("features", ",".join(PRUNED_FEATURE_COLS))
    mlflow.log_metric("cv_mean_wmae_reference", study.best_value)

    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="xgboost_pipeline",
        registered_model_name="walmart-xgboost-store-sales",
        serialization_format="cloudpickle",
    )

    print("Logged and registered as 'walmart-xgboost-store-sales'")
    print(f"Reference CV WMAE: {study.best_value:.2f}")

2026/07/11 19:33:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 19:33:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'walmart-xgboost-store-sales'.
2026/07/11 19:34:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-xgboost-store-sales, version 1
Created version '1' of model 'walmart-xgboost-store-sales'.


Logged and registered as 'walmart-xgboost-store-sales'
Reference CV WMAE: 2432.06
🏃 View run XGBoost_Final_Pipeline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1/runs/c7a078dc92a44c45b25c48a9b1f6f270
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/1


In [11]:
test_raw = test[["Store", "Dept", "Date", "IsHoliday"]]

test_preds = final_pipeline.predict(test_raw)
test_preds = np.clip(test_preds, 0, None)

submission = pd.DataFrame({
    "Id": test["Store"].astype(str) + "_" + test["Dept"].astype(str) + "_" + test["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_preds,
})

submission.to_csv("submission_xgboost.csv", index=False)
print(submission.shape)
submission.head()

(115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,40448.820312
1,1_1_2012-11-09,20543.042969
2,1_1_2012-11-16,21516.287109
3,1_1_2012-11-23,21224.658203
4,1_1_2012-11-30,26529.964844


In [12]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_xgboost.csv -m "XGBoost baseline, tuned, WMAE~2432 CV"

100% 2.87M/2.87M [00:00<00:00, 12.3MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting